# Inspect SEG-Y Outputs and Trace Spectrum

This notebook previews a SEG-Y image and computes a spectrum from a processed trace.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import segyio

from marine_sbp_asd2segy import amplitude_spectrum, traces_from_idx

segy_path = Path("outputs/line.asd.acf_envelope.sgy")
idx_path = Path("path/to/line.asd.acf.idx")
output_epsg = 32634


## Load a SEG-Y File

In [ ]:
def load_segy(path: Path):
    with segyio.open(str(path), ignore_geometry=True, strict=False, endian="big") as f:
        traces = segyio.collect(f.trace[:])
        data = traces.T.astype(np.float32, copy=False)
        dt_us = int(f.bin[segyio.BinField.Interval]) or 1000
    t_s = np.arange(data.shape[0]) * dt_us / 1_000_000.0
    return data, t_s

if segy_path.exists():
    data, t_s = load_segy(segy_path)
    print(data.shape)
else:
    print(f"Set segy_path to an existing SEG-Y file: {segy_path}")


## Plot an Amplitude Image

In [ ]:
if segy_path.exists():
    clip = np.nanpercentile(np.abs(data), 99.5)
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.imshow(
        data,
        aspect="auto",
        cmap="gray",
        vmin=-clip,
        vmax=clip,
        extent=[0, data.shape[1] - 1, t_s[-1], t_s[0]],
    )
    ax.set_xlabel("Trace #")
    ax.set_ylabel("Time (s)")
    ax.set_title(segy_path.name)
    plt.show()


## Spectrum from a Processed Trace

In [ ]:
if idx_path.exists():
    traces = traces_from_idx(idx_path, output_epsg=output_epsg, output="full_waveform")
    freq_hz, amp = amplitude_spectrum(traces[0].data, traces[0].dt)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(freq_hz, amp)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Amplitude")
    ax.set_title("First trace spectrum")
    ax.grid(True)
    plt.show()
else:
    print(f"Set idx_path to an existing .idx file: {idx_path}")
